In [1]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input

# 🔹 Paths
train_dir = r"D:\project\ML Project\smart-eye-diagnosis\data\raw\train"
val_dir   = r"D:\project\ML Project\smart-eye-diagnosis\data\raw\valid"
test_dir  = r"D:\project\ML Project\smart-eye-diagnosis\data\raw\test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 🔥 Train generator (with augmentation)

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

# ❌ No augmentation for val/test
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_data = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

test_data = val_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("Class indices:", train_data.class_indices)

# 🧠 MODEL (EfficientNetB0)
base_model = keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# base_model.trainable = False  # Freeze base
base_model.trainable = True

# Freeze only early layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)

output = layers.Dense(4, activation='softmax')(x)

model = keras.Model(inputs=base_model.input, outputs=output)

# ⚙️ Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ⚖️ Class weights (IMPORTANT)
class_weights = {
    0: 1.0,  # immature
    1: 1.0,  # mature
    2: 1.0,  # normal
    3: 2.5   # pterygium 🔥
}

# 🔥 Callbacks (VERY IMPORTANT)
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        "models/best_model.h5",
        monitor='val_accuracy',
        save_best_only=True
    )
]

# 🏋️ TRAIN
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    class_weight=class_weights,
    callbacks=callbacks
)
print(train_data.class_indices)

# 📊 EVALUATE
test_loss, test_acc = model.evaluate(test_data)
print("\nTest Accuracy:", test_acc)

# 💾 SAVE FINAL MODEL
model.save("models/final_model.h5")

Found 1977 images belonging to 4 classes.
Found 808 images belonging to 4 classes.
Found 763 images belonging to 4 classes.
Class indices: {'immature': 0, 'mature': 1, 'normal': 2, 'pterygium': 3}


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,219,175 (16.09 MB)

 Trainable params: 1,663,204 (6.34 MB)

...

62/62 ━━━━━━━━━━━━━━━━━━━━ 58s 943ms/step - accuracy: 0.9848 - loss: 0.0564 - val_accuracy: 0.9963 - val_loss: 0.0194
Epoch 9/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 602ms/step - accuracy: 0.9913 - loss: 0.0464

62/62 ━━━━━━━━━━━━━━━━━━━━ 49s 744ms/step - accuracy: 0.9863 - loss: 0.0555 - val_accuracy: 0.9975 - val_loss: 0.0154
Epoch 10/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 43s 690ms/step - accuracy: 0.9884 - loss: 0.0431 - val_accuracy: 0.9975 - val_loss: 0.0145
Epoch 11/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 553ms/step - accuracy: 0.9940 - loss: 0.0316

62/62 ━━━━━━━━━━━━━━━━━━━━ 43s 700ms/step - accuracy: 0.9929 - loss: 0.0326 - val_accuracy: 0.9988 - val_loss: 0.0082
Epoch 12/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 44s 703ms/step - accuracy: 0.9934 - loss: 0.0341 - val_accuracy: 0.9988 - val_loss: 0.0063
Epoch 13/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 52s 844ms/step - accuracy: 0.9884 - loss: 0.0387 - val_accuracy: 0.9975 - val_loss: 0.0071
Epoch 14/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 0s 565ms/step - accuracy: 0.9966 - loss: 0.0218

62/62 ━━━━━━━━━━━━━━━━━━━━ 44s 713ms/step - accuracy: 0.9960 - loss: 0.0218 - val_accuracy: 1.0000 - val_loss: 0.0034
Epoch 15/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 104s 2s/step - accuracy: 0.9939 - loss: 0.0183 - val_accuracy: 1.0000 - val_loss: 0.0028
Epoch 16/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 57s 874ms/step - accuracy: 0.9970 - loss: 0.0181 - val_accuracy: 1.0000 - val_loss: 0.0033
Epoch 17/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 48s 766ms/step - accuracy: 0.9985 - loss: 0.0121 - val_accuracy: 1.0000 - val_loss: 0.0027
Epoch 18/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 45s 725ms/step - accuracy: 0.9980 - loss: 0.0159 - val_accuracy: 0.9988 - val_loss: 0.0033
Epoch 19/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 45s 716ms/step - accuracy: 0.9970 - loss: 0.0150 - val_accuracy: 0.9988 - val_loss: 0.0021
Epoch 20/20
62/62 ━━━━━━━━━━━━━━━━━━━━ 46s 739ms/step - accuracy: 0.9970 - loss: 0.0148 - val_accuracy: 1.0000 - val_loss: 0.0014
{'immature': 0, 'mature': 1, 'normal': 2, 'pterygium': 3}
24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 335ms/step - acc


Test Accuracy: 1.0


In [6]:
import json

with open("models/history.json", "w") as f:
    json.dump(history.history, f)